In [2]:
import numpy as np
import pandas as pd
import requests

In [5]:
import truststore; truststore.inject_into_ssl()
import requests

r = requests.get("https://www.deribit.com/api/v2/public/ticker",
                 params={"instrument_name": "BTC-PERPETUAL"})
print(r.json()["result"])

{'timestamp': 1777404061504, 'state': 'open', 'stats': {'high': 77466.0, 'low': 75658.0, 'price_change': -0.9427, 'volume': 4659.19715003, 'volume_usd': 355596460.0, 'volume_notional': 355596460.0}, 'index_price': 76083.79, 'instrument_name': 'BTC-PERPETUAL', 'last_price': 76080.5, 'settlement_price': 76843.57, 'min_price': 74938.5, 'max_price': 77221.5, 'open_interest': 972596890, 'mark_price': 76079.99, 'interest_value': -0.013353143865366795, 'best_ask_price': 76080.5, 'best_bid_price': 76080.0, 'estimated_delivery_price': 76083.79, 'best_ask_amount': 314700.0, 'best_bid_amount': 6270.0, 'current_funding': 0.0, 'funding_8h': -1.4e-07}


In [8]:
requests.get("https://www.deribit.com/api/v2/public/get_book_summary_by_currency",
             params={"currency": "BTC", "kind": "option"}).json()

{'jsonrpc': '2.0',
 'result': [{'high': 0.016,
   'low': 0.0125,
   'last': 0.016,
   'instrument_name': 'BTC-30APR26-77500-P',
   'bid_price': 0.0205,
   'ask_price': 0.0235,
   'open_interest': 52.1,
   'mark_price': 0.02214434,
   'creation_timestamp': 1777404145005,
   'price_change': 18.5185,
   'interest_rate': 0,
   'volume': 3.0,
   'mark_iv': 38.43,
   'underlying_price': 76090.65,
   'underlying_index': 'index_price',
   'estimated_delivery_price': 76090.54,
   'base_currency': 'BTC',
   'quote_currency': 'BTC',
   'volume_usd': 3002.5,
   'mid_price': 0.022},
  {'high': None,
   'low': None,
   'last': None,
   'instrument_name': 'BTC-2MAY26-67000-C',
   'bid_price': 0.1105,
   'ask_price': 0.129,
   'open_interest': 0.0,
   'mark_price': 0.11948067,
   'creation_timestamp': 1777404145005,
   'price_change': None,
   'interest_rate': 0,
   'volume': 0.0,
   'mark_iv': 61.26,
   'underlying_price': 76090.65,
   'underlying_index': 'index_price',
   'estimated_delivery_price':

In [ ]:
# Verify what REST endpoints actually return for an option.
# Compares get_book_summary_by_currency (chain-level) vs public/ticker (per-instrument).

import requests, pandas as pd

# 1) Pick a real, currently-listed BTC option (front-expiry, near-ATM-ish).
summary = requests.get(
    "https://www.deribit.com/api/v2/public/get_book_summary_by_currency",
    params={"currency": "BTC", "kind": "option"},
).json()["result"]
print(f"book_summary returned {len(summary)} options for BTC")

# Take the first option that has both a bid and an ask quoted, so the IV fields are populated.
opt = next((s for s in summary if s.get("bid_price") and s.get("ask_price")), summary[0])
name = opt["instrument_name"]
print(f"sample instrument: {name}\n")

# 2) get_book_summary_by_currency fields
print("=== get_book_summary_by_currency fields ===")
for k in sorted(opt.keys()):
    v = opt[k]
    print(f"  {k!s:<28} = {v!r}")

# 3) public/ticker fields for the same instrument
ticker = requests.get(
    "https://www.deribit.com/api/v2/public/ticker",
    params={"instrument_name": name},
).json()["result"]
print("\n=== public/ticker fields ===")
for k in sorted(ticker.keys()):
    v = ticker[k]
    print(f"  {k!s:<28} = {v!r}")

# 4) Diff: which fields exist in ticker but not in book_summary?
extra_in_ticker = sorted(set(ticker.keys()) - set(opt.keys()))
extra_in_summary = sorted(set(opt.keys()) - set(ticker.keys()))
print("\n=== diff ===")
print(f"only in ticker:       {extra_in_ticker}")
print(f"only in book_summary: {extra_in_summary}")

# 5) Specifically confirm the size question.
print("\n=== sizes ===")
print(f"book_summary has best_bid_amount? {'best_bid_amount' in opt}")
print(f"book_summary has best_ask_amount? {'best_ask_amount' in opt}")
print(f"ticker       has best_bid_amount? {'best_bid_amount' in ticker} -> {ticker.get('best_bid_amount')}")
print(f"ticker       has best_ask_amount? {'best_ask_amount' in ticker} -> {ticker.get('best_ask_amount')}")
